In [ ]:
# ============================================================
# CELL 0 — Environment Setup (run this first, every session)
# ============================================================
# USE_DRIVE = True  → Google Colab + Google Drive
# USE_DRIVE = False → Local machine  ← DEFAULT
# ============================================================

import os, sys
from pathlib import Path

USE_DRIVE = False  # ← change to True only on Colab + Drive

# ── Auto-detect Colab ────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Resolve BASE (repo root) ─────────────────────────────────────────────────
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/retail-demand-forecasting')
elif IN_COLAB:
    BASE = Path('/content/dl-assignment')
else:
    # Local: CWD is either repo root or notebooks/ — handle both
    _cwd = Path(os.getcwd())
    BASE = _cwd.parent if _cwd.name == 'notebooks' else _cwd

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH      = BASE / 'data' / 'm5' / 'extracted'
PROCESSED_PATH = BASE / 'data' / 'm5' / 'processed'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

CALENDAR_PATH = DATA_PATH / 'calendar.csv'
PRICES_PATH   = DATA_PATH / 'sell_prices.csv'
SALES_PATH    = DATA_PATH / 'sales_train_validation.csv'

# ── Add src/ to Python path ───────────────────────────────────────────────────
SRC_PATH = str(BASE / 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Environment   : {"Google Colab" if IN_COLAB else "Local"}')
print(f'USE_DRIVE     : {USE_DRIVE}')
print(f'BASE          : {BASE}')
print(f'Device        : {device}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
print(f'DATA_PATH     : {DATA_PATH}')
print(f'DATA exists?  : {DATA_PATH.exists()}')
if not DATA_PATH.exists():
    print()
    print('[WARNING] DATA_PATH not found! Expected M5 CSVs in:')
    print(f'          {DATA_PATH}')


# 05 — GRU Demand Forecasting

This notebook trains and evaluates the GRU model. It uses the shared pipeline in `src/`, so the GRU gets the same data, split, features and metrics as the other models.

- **Task:** use the last 28 days to forecast demand for the next 7 days (direct multi-step regression).
- **Data:** a stratified 10% sample of M5 series by `store_id × cat_id` (seed 42).
- **Split:** by time. Train is days 1–1339, validation is days 1340–1626 and test is days 1627–1913.
- **Loss:** MSE on demand divided by the series' training-period mean demand.
- **Test rule:** tuning uses only validation data. The test period is used once, in Section 6.

In [ ]:
# Repo root for code and configs (on Colab + Drive, BASE points to the Drive data folder)
REPO = Path('/content/dl-assignment') if IN_COLAB else BASE
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

import itertools
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from train import load_config, train_model, get_device
from features import prepare_dataset, FEATURE_NAMES
from dataset import split_indices
from baselines import evaluate_baselines, window_arrays, BASELINES
from metrics import regression_metrics

RESULTS = BASE / 'results'
FIGURES = RESULTS / 'figures'
METRICS = RESULTS / 'metrics'
for folder in (FIGURES, METRICS):
    folder.mkdir(parents=True, exist_ok=True)

cfg = load_config(REPO / 'configs' / 'gru.yaml', REPO / 'configs' / 'experiment.yaml')
print('Training device:', get_device(cfg['training']['device']))

## 1. Build or load the feature dataset

The first run builds the features and saves a cache. Later runs, and the other model notebooks, load the same cache.

In [ ]:
data = prepare_dataset(
    DATA_PATH,
    cfg['data'],
    cache_path=PROCESSED_PATH / 'features.npz',
)
indices = split_indices(data, cfg['data'])

print('Features array :', data['features'].shape, '(series, days, features)')
print('Feature names  :', FEATURE_NAMES)
print('Windows        :', {split: len(index) for split, index in indices.items()})
data['series'].groupby(['store_id', 'cat_id']).size().unstack()

## 2. Baselines on validation

The GRU must beat these simple forecasts to be useful.

In [ ]:
baseline_val = pd.DataFrame(
    evaluate_baselines(data['sales'], indices['val'], cfg['data'])
).T
baseline_val

## 3. Hyperparameter search (validation only)

Each run stops early on validation loss. We choose the config with the lowest validation WAPE.

In [ ]:
grid = cfg['tuning']
tuning_rows = []

for hidden_size, num_layers, dropout in itertools.product(
    grid['hidden_size'], grid['num_layers'], grid['dropout']
):
    run_cfg = json.loads(json.dumps(cfg))
    run_cfg['model'].update(hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)

    _, result, _ = train_model(run_cfg, data, seed=cfg['seeds'][0], output_dir=RESULTS)

    tuning_rows.append({
        'hidden_size': hidden_size,
        'num_layers': num_layers,
        'dropout': dropout,
        'parameters': result['parameters'],
        'best_epoch': result['best_epoch'],
        'train_seconds': result['train_seconds'],
        **{f'val_{key}': value for key, value in result['val'].items()},
    })

tuning = pd.DataFrame(tuning_rows).sort_values('val_wape').reset_index(drop=True)
tuning.to_csv(METRICS / 'gru_tuning.csv', index=False)
tuning

In [ ]:
best = tuning.iloc[0]
best_model_cfg = {
    'name': 'gru',
    'hidden_size': int(best['hidden_size']),
    'num_layers': int(best['num_layers']),
    'dropout': float(best['dropout']),
}
print('Selected config:', best_model_cfg)

## 4. Final training with 3 seeds and a single test evaluation

We retrain the selected config with 3 seeds. We report the mean and standard deviation to show training stability.

In [ ]:
final_cfg = json.loads(json.dumps(cfg))
final_cfg['model'] = best_model_cfg

final_results, final_histories = [], {}
for seed in cfg['seeds']:
    model, result, history = train_model(
        final_cfg, data, seed=seed, evaluate_test=True,
        run_name=f'gru_final_seed{seed}', output_dir=RESULTS,
    )
    final_results.append(result)
    final_histories[seed] = pd.DataFrame(history)

rows = []
for result in final_results:
    rows.append({
        'seed': result['seed'],
        'parameters': result['parameters'],
        'epochs_run': result['epochs_run'],
        'train_seconds': result['train_seconds'],
        'inference_ms_per_1000_windows': result['inference_ms_per_1000_windows'],
        **{f'val_{k}': v for k, v in result['val'].items()},
        **{f'test_{k}': v for k, v in result['test'].items()},
    })

final_table = pd.DataFrame(rows)
final_table.to_csv(METRICS / 'gru_final_seeds.csv', index=False)
final_table.agg(['mean', 'std']).T

## 5. Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for seed, history in final_histories.items():
    ax.plot(history['epoch'], history['train_loss'], label=f'train (seed {seed})')
    ax.plot(history['epoch'], history['val_loss'], linestyle='--', label=f'validation (seed {seed})')

ax.set_title('GRU learning curves')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (scaled demand)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'gru_learning_curves.png', dpi=150)
plt.show()

## 6. Test results against the baselines

In [ ]:
baseline_test = pd.DataFrame(
    evaluate_baselines(data['sales'], indices['test'], cfg['data'])
).T

gru_test = pd.DataFrame([r['test'] for r in final_results]).mean().rename(f"gru (mean of {len(final_results)} seeds)")
comparison = pd.concat([baseline_test, gru_test.to_frame().T])
comparison.to_csv(METRICS / 'gru_test_comparison.csv')
comparison

In [ ]:
by_horizon = pd.DataFrame(final_results[0]['test_by_horizon'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_horizon['horizon_day'], by_horizon['mae'], marker='o')
ax.set_title('GRU test MAE by forecast day (seed %d)' % final_results[0]['seed'])
ax.set_xlabel('Days ahead')
ax.set_ylabel('MAE (units)')
plt.tight_layout()
plt.savefig(FIGURES / 'gru_mae_by_horizon.png', dpi=150)
plt.show()
by_horizon

## 7. Error analysis by demand pattern

We group series by their share of zero-demand days and by category. This shows where the GRU is weak.

In [ ]:
predictions = np.load(RESULTS / 'predictions' / f"gru_final_seed{cfg['seeds'][0]}_test.npz")
y_pred, y_true, rows = predictions['y_pred'], predictions['y_true'], predictions['rows']

train_sales = data['sales'][:, :cfg['data']['train_end']]
active_days = np.arange(cfg['data']['train_end'])[None, :] >= data['first_valid'][:, None]
zero_share = ((train_sales == 0) & active_days).sum(axis=1) / active_days.sum(axis=1)

series_info = data['series'].copy()
series_info['zero_share'] = zero_share
series_info['pattern'] = pd.cut(
    zero_share, bins=[-0.01, 0.3, 0.7, 1.0],
    labels=['frequent (<30% zeros)', 'mixed (30-70%)', 'intermittent (>70%)'],
)

window_info = series_info.iloc[rows[:, 0]].reset_index(drop=True)
error_rows = []
for column in ['pattern', 'cat_id']:
    for group, mask in window_info.groupby(column, observed=True).groups.items():
        error_rows.append({
            'group_by': column,
            'group': group,
            'windows': len(mask),
            **regression_metrics(y_true[mask], y_pred[mask]),
        })

errors = pd.DataFrame(error_rows)
errors.to_csv(METRICS / 'gru_error_by_group.csv', index=False)
errors

In [ ]:
# One example series per demand pattern
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, pattern in zip(axes, series_info['pattern'].cat.categories):
    candidates = series_info.index[series_info['pattern'] == pattern]
    if len(candidates) == 0:
        continue
    series = candidates[0]
    mask = rows[:, 0] == series
    order = np.argsort(rows[mask, 1])
    days = (rows[mask, 1][order][:, None] + np.arange(cfg['data']['horizon'])).ravel()

    ax.plot(days, y_true[mask][order].ravel(), label='actual')
    ax.plot(days, y_pred[mask][order].ravel(), label='GRU forecast')
    info = series_info.loc[series]
    ax.set_title(f"{info['item_id']} @ {info['store_id']} — {pattern}")
    ax.set_ylabel('Units')
    ax.legend()

axes[-1].set_xlabel('Day index')
plt.tight_layout()
plt.savefig(FIGURES / 'gru_forecast_examples.png', dpi=150)
plt.show()

## 8. Points for the report

- Compare the GRU with the baselines. Does it beat seasonal naive and the 28-day moving average on every metric?
- Compare the GRU with the LSTM. A GRU has 3 gates and an LSTM has 4, so the GRU has about 25% fewer parameters. Is the accuracy similar?
- Use the learning curves to check for overfitting. Look at the gap between training and validation loss, and at the best epoch.
- Use the seed standard deviation to discuss training stability.
- sMAPE is high for intermittent series because many days have 0 demand. So we use WAPE as the main metric.
- Possible improvements: add known future calendar features for the 7 target days, a Tweedie loss for zero-heavy demand, and item or store embeddings.